# ddc — MS vs. Control Analysis

Analysis of participant data comparing **MS** patients vs. **Controls**.

## Workflow
1. Load data (raw + preprocessed)
2. Demographics table
3. Age- & gender-matched dataframe (matched to MS participants)
4. Predictor variable list
5. Data exploration (distributions, missing values, interpolation)
6. Correlation between predictors (across group + within group)
7. Elastic net & multinomial logistic regression (notes)
8. Elastic net model (controls dummy-coded to 0)

> **Kernel:** make sure `ddc-env` is selected (Kernel -> Change kernel).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid")

print("Setup complete.")

In [ ]:
# ---- LOAD DATA ----
# Raw participants (age, gender, group)
raw = pd.read_csv("../data/demo_participants.csv")

# NOTE: load your preprocessed/participant data here when you have it,
# e.g. preprocessed = pd.read_csv("../data/preprocessed.csv")
print(raw.shape)
raw.head()

## 1. Demographics table
A clean table of participant counts / summary statistics by group.

In [ ]:
# Demographics table by group
demo_table = raw.groupby("group").agg(
    n=("participant_id", "count"),
    age_mean=("age", "mean"),
    age_sd=("age", "std"),
    n_female=("gender", lambda s: (s == "F").sum()),
    n_male=("gender", lambda s: (s == "M").sum()),
    pct_female=("gender", lambda s: (s == "F").mean() * 100),
).round(2)
demo_table

## 2. Age- & gender-matched dataframe (matched to MS participants)
The **MS group is the smallest**, so we match controls to MS participants on
gender, picking the closest age (absolute difference). Optionally allow exact
age first, then nearest available.


In [ ]:
def match_controls(df, match_col="group", ms_label="MS", ctrl_label="Control",
                    gender_col="gender", age_col="age", seed=42):
    ms = df[df[match_col] == ms_label]
    ctrls = df[df[match_col] == ctrl_label].copy()

    matched = []
    pool = ctrls.copy()
    rng = np.random.default_rng(seed)
    for _, p in ms.iterrows():
        if pool.empty:
            break
        candidates = pool[pool[gender_col] == p[gender_col]]
        if candidates.empty:
            continue  # no control of matching gender available
        candidates = candidates.assign(_diff=(candidates[age_col] - p[age_col]).abs())
        best = candidates.sort_values("_diff").iloc[0]
        matched.append(best)
        pool = pool.drop(best.name)

    matched_df = pd.DataFrame(matched).drop(columns="_diff", errors="ignore")
    matched_df[match_col] = ctrl_label
    return pd.concat([ms, matched_df], ignore_index=True).sort_values(match_col)


matched = match_controls(raw)
print("Matched shape:", matched.shape)
print(matched.groupby("group").agg(n=("participant_id", "count"),
                                   age_mean=("age", "mean")))
matched

## 3. Predictor variables
Pick the predictor variables you care about **from the preprocessed data**.
Below is a placeholder list — replace with your real variable names.


In [ ]:
# TODO: point this at your preprocessed data
# preprocessed = pd.read_csv("../data/preprocessed.csv")

# Replace with the names of the predictors you are interested in
PREDICTORS = [
    "age",
    # "sex_male", "disease_duration", "edss", "t2_lesion_count", ...
]

# Build X for analysis (drop ids / outcome / non-numeric)
X = matched[PREDICTORS].copy()
print("Predictors:", list(X.columns))
print("Shape:", X.shape)
X.head()

## 4. Data exploration
- Distributions of each variable (histograms, by group)
- Missing values? How to handle them? (drop / impute / interpolate)
- Correlation heatmap (visual)


In [ ]:
# Distribution by group
if len(PREDICTORS) <= 6:
    ncols = 3
    nrows = int(np.ceil(len(PREDICTORS) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4, nrows * 3))
    axes = np.atleast_2d(axes).ravel()
    for ax, col in zip(axes, PREDICTORS):
        for g in matched["group"].unique():
            sns.histplot(matched.loc[matched["group"] == g, col],
                         ax=ax, label=g, alpha=0.5, kde=True)
        ax.set_title(col)
    for ax in axes[len(PREDICTORS):]:
        ax.set_visible(False)
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("Too many predictors to plot all at once.")

# Missing values
print("\nMissing values per column:")
print(matched[PREDICTORS].isna().sum())

## 5. Correlation between predictors
- **Across group**: one correlation plot using all participants
- **Within group**: one plot per group → **total of 5 plots** (1 across + 4 groups)


In [ ]:
groups = matched["group"].unique()
print("Groups:", groups)

# (a) Across group — all participants
sns.clustermap(matched[PREDICTORS].corr(), annot=True, cmap="coolwarm", center=0)
plt.show()

# (b) Within group — one plot per group
for g in groups:
    gx = matched[matched["group"] == g][PREDICTORS].corr()
    plt.figure(figsize=(6, 5))
    sns.heatmap(gx, annot=True, cmap="coolwarm", center=0, vmin=-1, vmax=1)
    plt.title(f"Correlations — {g}")
    plt.tight_layout()
    plt.show()

## 6. Elastic net & multinomial logistic regression — notes
Add your research notes here (what we learned from the reading).

- **Elastic net**: linear/GLM model combining L1 (Lasso) + L2 (Ridge) penalties. Good when you have many predictors, correlated predictors, and want feature selection while keeping some stability.
  - Loss: `L = loss(y, Xw) + alpha * rho * ||w||_1 + 0.5 * alpha * (1 - rho) * ||w||_2^2`
  - `l1_ratio = rho` mixes the penalties; `alpha` controls overall strength.
  - **Pros:** handles correlated predictors, shrinks + selects features, works with p >> n.
  - **Cons:** biased coefficient estimates (shrinkage), needs tuning of 2 hyperparameters, less interpretable than unregularized models.

- **Multinomial logistic regression**: generalization of logistic regression to **multiple classes** (softmax). Good when the outcome has >2 unordered categories. Regularized (`LogisticRegression` with `solver="lbfgs"` / `liblinear`, `penalty="l1"`/`l2`/`elasticnet`).

- **Why good for us:** if the outcome is multi-class (e.g., MS subtypes) use multinomial; if binary MS vs. Control, binary elastic-net logistic regression works.


## 7. Elastic net model
- `X` = predictors (numeric; categoricals dummy-coded)
- `y` = outcome, with **Controls dummy-coded to 0** (MS = 1, Control = 0)
- Standardize features, tune `alpha`/`l1_ratio` with cross-validation.


In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import ElasticNet, LogisticRegression

# Outcome: Controls dummy-coded to 0, MS = 1
y = (matched["group"] == "MS").astype(int).values

print("Class balance:")
print(pd.Series(y, name="y").value_counts().sort_index())

# Feature matrix (ensure numeric predictors only)
Xf = X.select_dtypes(include=[np.number]).copy()
print("Feature matrix:", Xf.shape)

# Pipeline: standardize -> elastic net logistic regression
# (use ElasticNet for regression; LogisticRegression for classification)
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(penalty="elasticnet", solver="saga",
                                 l1_ratio=0.5, max_iter=5000, random_state=42)),
])

grid = {
    "model__l1_ratio": [0.2, 0.5, 0.8],
    "model__C": [0.01, 0.1, 1.0, 10.0],
}

X_train, X_test, y_train, y_test = train_test_split(
    Xf, y, test_size=0.3, stratify=y, random_state=42)

search = GridSearchCV(pipe, grid, cv=3, scoring="roc_auc")
search.fit(X_train, y_train)

print("Best params:", search.best_params_)
print("Best CV ROC-AUC:", round(search.best_score_, 3))
print("Test ROC-AUC:  ", round(search.score(X_test, y_test), 3))

coefs = pd.Series(search.best_estimator_.named_steps["model"].coef_.ravel(),
                  index=Xf.columns).sort_values()
print("\nCoefficients (sorted):")
print(coefs)

In [ ]:
# ---- SESSION INFO ----
import sys
print(sys.version)
from platform import python_version
print("Python:", python_version())
print("Pandas:", pd.__version__)
print("Sklearn:", __import__("sklearn").__version__)